# 6.2 环境搭建与工具准备

本节准备好实验所需的全部基础设施：
1. AirSim 场景配置与启动
2. 无人机控制工具函数
3. LLM 调用封装

所有代码封装在 `airsim_tools.py` 中，后续实验直接导入使用。

## 6.2.1 配置 AirSim 场景

本章提供了两个 settings.json 配置文件：
- `1-6-settings.json`：2架无人机（Drone1、Drone2），用于第1-6节
- `7-settings.json`：3架无人机（Drone1、Drone2、Drone3），用于第7节层级协同

使用 `apply_settings()` 函数可以一键切换配置并重启 AirSim：

## 6.2.2 安装依赖

本章只需要两个 Python 包（不需要 langchain/langgraph）：

> 第5-6节使用 CrewAI 框架时需要额外安装 `crewai`

In [5]:
# 方式1：一键切换配置并重启 AirSim（推荐）
from airsim_tools import apply_settings

# 第1-6节用2架无人机
apply_settings("1-6-settings.json")

# 第7节用3架无人机时，改成：
# apply_settings("7-settings.json")

已复制 1-6-settings.json → /root/Documents/AirSim/settings.json
正在重启 AirSim，等待 10 秒...
AirSim 已重启


## 6.2.3 连接验证

In [6]:
import sys
sys.path.append('../external-libraries')

import airsim

# 连接到 AirSim
client = airsim.MultirotorClient()
client.confirmConnection()

# 列出所有无人机
vehicles = client.listVehicles()
print(f"连接成功！发现无人机: {vehicles}")
# 预期输出: 连接成功！发现无人机: ['Drone1', 'Drone2']

Connected!
Client Ver:1 (Min Req: 1), Server Ver:1 (Min Req: 1)

连接成功！发现无人机: ['Drone1', 'Drone2']


## 6.2.4 工具模块：airsim_tools.py

我们将所有工具函数封装在 `airsim_tools.py` 中。它包含两部分：

### Part 1：AirSim 无人机控制函数

三个核心函数，全部是**纯Python函数**，没有任何框架依赖：

| 函数 | 功能 | 参数 |
|------|------|------|
| `takeoff(client, drone_id)` | 起飞 | AirSim客户端, 无人机名称 |
| `fly_to(client, drone_id, x, y, z)` | 飞到指定坐标 | 客户端, 名称, NED坐标 |
| `get_state(client, drone_id)` | 获取当前位置 | 客户端, 名称 |

> 注意：AirSim 使用 NED 坐标系，z 为负表示向上（例如 z=-10 代表 10米高度）

### Part 2：LLM 调用封装

一个 `call_llm(prompt)` 函数，使用 OpenAI SDK 调用豆包（Doubao）模型：

```python
def call_llm(prompt, system="你是一个无人机任务规划助手，请简洁回答。"):
    client = OpenAI(base_url=LLM_BASE_URL, api_key=LLM_API_KEY)
    response = client.chat.completions.create(
        model=LLM_MODEL,
        temperature=0.1,
        messages=[{"role": "system", "content": system},
                  {"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content
```

就这么简单——3行核心代码，发一个请求、拿回文本。

## 6.2.5 快速测试

### 测试 LLM 调用

In [7]:
from airsim_tools import call_llm

# 测试 LLM 是否正常工作
result = call_llm("请用一句话解释什么是无人机集群协同。")
print(result)

无人机集群协同是指多架无人机通过实时信息交互、分布式自主决策与动态分工配合，共同完成单架无人机难以高效执行的侦察、攻击、运输、应急救援等复杂任务的作业模式。


### 测试 AirSim 控制

In [8]:
from airsim_tools import connect_airsim, takeoff, fly_to, get_state

# 连接
client = connect_airsim()

# 让 Drone1 起飞并飞到 (5, 0, -5)
takeoff(client, "Drone1")
fly_to(client, "Drone1", 5, 0, -5)

# 查看位置
pos = get_state(client, "Drone1")
print(f"Drone1 当前位置: {pos}")

Connected!
Client Ver:1 (Min Req: 1), Server Ver:1 (Min Req: 1)

已连接到 AirSim 模拟器
[Drone1] 起飞完成
[Drone1] 已到达 (5, 0, -5)
Drone1 当前位置: {'x': 5.17, 'y': -0.0, 'z': -4.98}


如果两个测试都通过，说明 LLM 和 AirSim 环境都已就绪，可以开始实验了。